# KL Divergence

**Audience:** students who have just met entropy, cross-entropy and Bayes' rule.

**What you will build:** a hands-on exploration of the Kullback--Leibler (KL)
divergence: its definition, non-negativity, asymmetry, link to cross-entropy and
maximum likelihood, and a closed-form formula for Gaussians.

## Learning objectives

1. compute KL divergence for discrete and continuous distributions;
2. prove and visualise that KL is non-negative and vanishes only for identical laws;
3. see why KL is a divergence, not a distance (it is asymmetric);
4. connect minimising KL to maximum likelihood and cross-entropy training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

rng = np.random.default_rng(0)
plt.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print("Setup complete")

## 1. Definition (discrete case)

For two distributions $p$ and $q$ on the same alphabet,

$$D_{\mathrm{KL}}(p\,\|\,q) = \sum_x p(x)\log\frac{p(x)}{q(x)}.$$

With natural logarithms the unit is **nats**; base 2 gives bits. By convention
$0\log(0/q)=0$, while $p(x)>0$ with $q(x)=0$ makes the divergence infinite: a
model that assigns zero probability to something the data actually shows is
infinitely bad.

In [ ]:
def bernoulli_kl(p, q):
    """KL(Bernoulli(p) || Bernoulli(q)) in nats."""
    if not 0.0 < p < 1.0 or not 0.0 < q < 1.0:
        raise ValueError("p and q must lie strictly between 0 and 1")
    return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))


p, q = 0.5, 0.1
value = bernoulli_kl(p, q)
print(f"KL(Bernoulli({p}) || Bernoulli({q})) = {value:.4f} nats")
print(f"scipy check: {scipy_entropy([p, 1 - p], [q, 1 - q]):.4f}")

## 2. Non-negativity (Gibbs' inequality)

Apply Jensen's inequality to the convex function $-\log$ with weights $p(x)$:

$$D_{\mathrm{KL}}(p\,\|\,q) = -\sum_x p(x)\log\frac{q(x)}{p(x)}
\;\geq\; -\log\sum_x p(x)\frac{q(x)}{p(x)} = -\log 1 = 0.$$

Equality requires $q(x)/p(x)$ to be constant on the support of $p$, which forces
$p=q$. So KL is always non-negative and vanishes only for identical laws.

In [ ]:
qs = np.linspace(0.01, 0.99, 400)
p0 = 0.5
kl_curve = np.array([bernoulli_kl(p0, q) for q in qs])

fig, ax = plt.subplots()
ax.plot(qs, kl_curve, lw=2)
ax.axvline(p0, color="tab:red", ls="--", label="q = p")
ax.set(xlabel="q", ylabel="KL(p || q)", title="KL is minimised at q = p")
ax.legend()
plt.show()

print(f"minimum near q = {qs[np.argmin(kl_curve)]:.3f}, value = {kl_curve.min():.2e}")

## 3. KL is not a distance

Nothing in the definition is symmetric: in general
$D_{\mathrm{KL}}(p\,\|\,q) \neq D_{\mathrm{KL}}(q\,\|\,p)$, and the triangle
inequality fails too. KL is a **divergence**, not a metric. It measures the extra
coding cost of using $q$ when the true law is $p$, and that cost depends on which
law is treated as the truth. The heat map below is not symmetric about the
diagonal.

In [ ]:
ps = np.linspace(0.02, 0.98, 200)
qs = np.linspace(0.02, 0.98, 200)
P, Q = np.meshgrid(ps, qs, indexing="ij")
KL = P * np.log(P / Q) + (1 - P) * np.log((1 - P) / (1 - Q))

fig, ax = plt.subplots()
im = ax.imshow(
    KL, origin="lower", extent=(qs[0], qs[-1], ps[0], ps[-1]),
    aspect="auto", cmap="viridis",
)
ax.plot([0.02, 0.98], [0.02, 0.98], color="w", ls="--", lw=1, label="p = q")
ax.set(xlabel="q", ylabel="p", title="KL(Bernoulli(p) || Bernoulli(q))")
ax.legend()
fig.colorbar(im, ax=ax, label="nats")
plt.show()

print(f"KL(p||q) = {bernoulli_kl(0.5, 0.1):.4f}")
print(f"KL(q||p) = {bernoulli_kl(0.1, 0.5):.4f}")

## 4. Cross-entropy = entropy + KL

$$H(p,q) = -\sum_x p(x)\log q(x) = H(p) + D_{\mathrm{KL}}(p\,\|\,q).$$

The entropy $H(p)$ does not depend on the model $q$, so minimising cross-entropy
is exactly the same as minimising KL. When $p$ is a one-hot target this is the
negative log-likelihood used to train a softmax classifier.

In [ ]:
def binary_entropy(p):
    return -p * np.log(p) - (1 - p) * np.log(1 - p)


p0 = 0.5
qs = np.linspace(0.01, 0.99, 400)
ce = -p0 * np.log(qs) - (1 - p0) * np.log(1 - qs)
kl = np.array([bernoulli_kl(p0, q) for q in qs])
h = binary_entropy(p0)
print(f"H(p) = {h:.4f} nats")
print(f"max |H(p,q) - H(p) - KL(p||q)| = {np.max(np.abs(ce - h - kl)):.2e}")

fig, ax = plt.subplots()
ax.plot(qs, ce, label="cross-entropy H(p,q)")
ax.plot(qs, kl, ls="--", label="KL(p||q)")
ax.axvline(p0, color="0.4", ls=":")
ax.set(xlabel="q", ylabel="nats", title="Cross-entropy and KL differ by H(p)")
ax.legend()
plt.show()

## 5. KL between two Gaussians

For $p=\mathcal{N}(\mu_0,\sigma_0^2)$ and $q=\mathcal{N}(\mu_1,\sigma_1^2)$,

$$D_{\mathrm{KL}}(p\,\|\,q) = \log\frac{\sigma_1}{\sigma_0}
+ \frac{\sigma_0^2 + (\mu_0-\mu_1)^2}{2\sigma_1^2} - \frac12.$$

The multivariate analogue uses log-determinants and a trace. Below we check the
closed form against a Monte-Carlo estimate of
$\mathrm{E}_{p}[\log p(x) - \log q(x)]$.

In [ ]:
def gaussian_kl(mu0, s0, mu1, s1):
    return np.log(s1 / s0) + (s0**2 + (mu0 - mu1) ** 2) / (2 * s1**2) - 0.5


def normal_logpdf(x, mu, s):
    return -0.5 * np.log(2 * np.pi * s**2) - (x - mu) ** 2 / (2 * s**2)


mu0, s0, mu1, s1 = 0.0, 1.0, 1.0, 1.5
exact = gaussian_kl(mu0, s0, mu1, s1)
samples = rng.normal(mu0, s0, size=400_000)
mc = np.mean(normal_logpdf(samples, mu0, s0) - normal_logpdf(samples, mu1, s1))
print(f"closed form  = {exact:.4f}")
print(f"Monte Carlo  = {mc:.4f}")
print(f"identical    = {gaussian_kl(0.0, 1.0, 0.0, 1.0):.4f}")

## 6. KL minimisation is maximum likelihood

Let $p_{\mathrm{data}}$ be the empirical distribution of the sample and
$p_\theta$ the model. Then

$$\arg\min_\theta D_{\mathrm{KL}}(p_{\mathrm{data}}\,\|\,p_\theta)
= \arg\max_\theta \sum_{i=1}^n \log p_\theta(x^{(i)}),$$

because the entropy of $p_{\mathrm{data}}$ is independent of $\theta$. Adding a
prior and maximising the posterior gives MAP: an $\ell_2$ penalty corresponds to
a Gaussian prior, and an $\ell_1$ penalty to a Laplace prior.

In [ ]:
data = rng.binomial(1, 0.7, size=200)
p_hat = float(np.clip(data.mean(), 1e-6, 1 - 1e-6))
theta_grid = np.linspace(0.05, 0.95, 400)
kl_grid = np.array([bernoulli_kl(p_hat, t) for t in theta_grid])
print(f"empirical p          = {p_hat:.3f}")
print(f"KL-minimising theta  = {theta_grid[np.argmin(kl_grid)]:.3f}")

fig, ax = plt.subplots()
ax.plot(theta_grid, kl_grid)
ax.axvline(p_hat, color="tab:red", ls="--", label="MLE")
ax.set(xlabel="theta", ylabel="KL(empirical || Bernoulli(theta))",
       title="Minimising KL recovers the MLE")
ax.legend()
plt.show()

## Exercises

1. Show numerically that `bernoulli_kl(p, q)` tends to $+\infty$ as $q\to0$
   with $p>0$.
2. Use `scipy.optimize.minimize_scalar` to minimise KL over $q$ and check that the
   optimum equals $p$.
3. Compute KL between two categorical distributions with three outcomes and
   verify it with `scipy.stats.entropy`.
4. For fixed variance, plot Gaussian KL as a function of the mean gap. What is the
   curvature at zero?
5. Explain why cross-entropy training with a one-hot target is maximum likelihood
   for the softmax classifier.